# Contrastive rewrite — qwen / humor

Two DPO arms differing only in where the **chosen** response comes from. Same base
model, same prompts, same rejected responses, same seed (123456), same hyperparameters.

| arm | chosen | status |
|---|---|---|
| baseline | GLM-4.5-Air writing from scratch with the constitution in context | already trained: `sdananya/qwen-2.5-7b-it-humor` |
| `_hybrid` | free generation on the 500 constitution prompts, qwen minimally editing its own response on the 1330 LIMA prompts | this notebook |

Only the hybrid arm needs a GPU. Cells are order-dependent: 4 creates the qwen
column, 8 needs it, 12 needs 8.

In [ ]:
import os

# HF_HOME must be set before anything imports huggingface_hub
os.environ["HF_TOKEN"]    = "hf_..."
os.environ["WANDB_TOKEN"] = "..."          # 40 chars, not the setup placeholder
os.environ["HF_USER"]     = "invi-bhagyesh"  # sdananya holds the baseline; this arm is yours
os.environ["HF_HOME"]     = "/workspace/.cache/huggingface"
os.chdir("/workspace/OpenCharacterTraining")

C, M, KEY, GPU = "humor", "qwen-2.5-7b-it", "qwen", "0"
BASELINE = "sdananya/qwen-2.5-7b-it-humor"

In [ ]:
!git fetch origin && git checkout contrastive-rewrite && git log --oneline -1
!python run_all.py --model $KEY --download-models

## 1. Baseline data

Idempotent — seeds the teacher responses from HF and generates qwen's rejected
responses only if they are not already local. This is also what the baseline arm
was trained on, so it doubles as a provenance check.

In [ ]:
!CUDA_VISIBLE_DEVICES=$GPU python run_data.py --stage dpo --model $M --constitution $C

## 2. Is the length gap real?

If chosen is systematically longer, DPO learns "longer = better" independent of
content. This number is how much the rewrite is worth — if the gap is small, the
premise is weaker than assumed and it is worth stopping here.

In [ ]:
import pandas as pd

d = pd.read_json(f"data/dpo/{M}/{C}.jsonl", lines=True)
c = d["chosen"].apply(lambda m: len(m[1]["content"]))
r = d["rejected"].apply(lambda m: len(m[1]["content"]))
print(f"{len(d)} pairs | chosen {c.mean():.0f} chars, rejected {r.mean():.0f}, "
      f"chosen longer in {(c > r).mean():.0%}")

## 3. Rewrite pass

qwen edits its own responses. The paper's conditional rewrite uses the same model as
the original, so there is no teacher style left to leak.

In [ ]:
!CUDA_VISIBLE_DEVICES=$GPU python -m character.distillation.teacher \
    --model $M --mode rewrite --student $M --constitution $C

## 4. Did it actually edit minimally?

Mostly `unchanged` means the trait is not landing and the rewrite prompt needs work.
Mostly dropped on length means qwen regenerated instead of editing — loosen
`--max-ratio` or use a larger rewriter.

In [ ]:
d = pd.read_json(f"data/distillation/{C}.jsonl", lines=True)
col = f"rewrite_{M}"
print(f"{d[col].notna().sum()} / {d[M].notna().sum()} rewrites kept\n")

row = d[d[col].notna()].iloc[0]
for k in ["prompt", M, col]:
    print(f"--- {k} ---\n{row[k][:400]}\n")

## 5. Hybrid arm data

In [ ]:
!CUDA_VISIBLE_DEVICES=$GPU python run_data.py --stage dpo --model $M --constitution $C \
    --chosen-source hybrid

## 6. Train the hybrid arm

Detached, so a kernel restart does not kill it. DeepSpeed checkpoints are ~30 GB
each at `--save-steps 100`; check `df -h` before starting.

In [ ]:
!df -h /workspace
!nohup env CUDA_VISIBLE_DEVICES=$GPU python run_all.py \
    --model $KEY --constitution $C --stage dpo --arm _hybrid > log_hybrid.txt 2>&1 &

In [ ]:
!tail -5 log_hybrid.txt
!df -h /workspace

## 7. Pull the baseline adapter to compare against

In [ ]:
from huggingface_hub import snapshot_download

p = snapshot_download(BASELINE, allow_patterns=["dpo-final/*"],
                      local_dir=f"/workspace/loras/{KEY}-distillation/{C}_baseline")
print(p)